# TsekTxt: test the 3 models (Colab, GPU optional)
Runtime > Change runtime type > T4 GPU (faster, not required). Run cells top to bottom.

In [ ]:
!git clone -q https://github.com/hansjio/tsektxt-model-training.git
%cd tsektxt-model-training
!pip -q install 'transformers>=4.41,<5' scikit-learn pandas pyyaml

## 1. Try any text (all 3 models, loaded from Hugging Face)

In [ ]:
!python src/predict.py --text 'Totoo ba na may dayaan sa eleksyon? Share this para malaman ng lahat!'

## 2. Overfitting check: train/val/test accuracy
Needs the splits. This rebuilds them from data/raw (a few minutes, CPU ok).

In [ ]:
!pip -q install ftfy datasets
!python src/clean_data.py --input data/raw/combined_dataset_clean.csv
!python src/make_splits.py
!python src/predict.py --split-gap

## 3. Evaluate on the Taglish test set
Upload `taglish_test.csv` with columns `id,text,label` (1 = credible, 0 = not credible). Never train on it.

In [ ]:
from google.colab import files
import os
up = files.upload()
os.makedirs('data/external', exist_ok=True)
os.replace(list(up)[0], 'data/external/taglish_test.csv')

In [ ]:
for m in ['roberta-tagalog','mbert','xlmr']:
    !python src/evaluate_external.py --model {m} --csv data/external/taglish_test.csv

## 4. Download results
Copy `results/*/external_metrics.json` and `external_predictions.csv` back to your repo.

In [ ]:
!zip -qr external_results.zip results/*/external_*
files.download('external_results.zip')